# DEAP Preprocessing Sensitivity - Subject-Independent MFMC

This notebook only loads prepared folders under `Processed_data/` and never re-runs preprocessing.

Expected environment:
- `conda activate MFMC`


In [ ]:
# Imports and plotting style

import json
import os
import random
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.grid": False,
    "grid.alpha": 0.2,
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
})

print("Run this notebook in the MFMC conda environment: conda activate MFMC")


In [ ]:
# Global paths and settings

PROJECT_ROOT = Path(os.environ.get("TAFFC_MFMC_ROOT", "/home/zhengdeyang/TAFFC_MFMC"))
SUPPLEMENT_DIR = PROJECT_ROOT / "MFMC" / "Supplement" / "Preprocess_sensitivity"
PROCESSED_ROOT = SUPPLEMENT_DIR / "Processed_data"
RESULTS_DIR = SUPPLEMENT_DIR / "Results" / "subject_indep"
FIGURES_DIR = SUPPLEMENT_DIR / "Figures" / "subject_indep"
MANIFEST_PATH = PROCESSED_ROOT / "manifest.csv"

BATCH_SIZE = 200
TOTAL_ITERATIONS = 20001
EVAL_INTERVAL = 500
RANDOM_SEED = 42
LEARNING_RATE_ENCODER = 0.0003
LEARNING_RATE_CLASSIFIER = 0.0003
BETA1 = 0.9
BETA2 = 0.999
COV_BETA = 0.5
USE_CLASS_BALANCING = True
SETTING_NAME_FILTER = None  # Example: ["baseline", "wl_10"]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed data root: {PROCESSED_ROOT}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"USE_CLASS_BALANCING={USE_CLASS_BALANCING}")


In [ ]:
# Helper functions

def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device(min_free_memory_mb: int = 12000) -> torch.device:
    """Select the CUDA GPU with the most free VRAM when enough headroom is available."""
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is available. Stop this notebook instead of running on CPU.")

    query = [
        "nvidia-smi",
        "--query-gpu=index,memory.free,memory.total,memory.used",
        "--format=csv,noheader,nounits",
    ]
    try:
        output = subprocess.check_output(query, encoding="utf-8")
    except Exception as exc:
        raise RuntimeError(f"Could not query GPU memory with nvidia-smi: {exc}") from exc

    gpu_stats = []
    for line in output.strip().splitlines():
        index, free_mem, total_mem, used_mem = [part.strip() for part in line.split(",")]
        gpu_stats.append({
            "index": int(index),
            "free": int(free_mem),
            "total": int(total_mem),
            "used": int(used_mem),
        })

    if not gpu_stats:
        raise RuntimeError("CUDA is available, but nvidia-smi returned no GPU rows.")

    suitable = [gpu for gpu in gpu_stats if gpu["free"] >= min_free_memory_mb]
    if not suitable:
        status = ", ".join(
            f"GPU {gpu['index']}: {gpu['free']} MB free / {gpu['total']} MB total"
            for gpu in gpu_stats
        )
        raise RuntimeError(
            f"OOM: no GPU has at least {min_free_memory_mb} MB free VRAM. Current status: {status}"
        )

    selected = max(suitable, key=lambda gpu: gpu["free"])
    device = torch.device(f"cuda:{selected['index']}")
    print(
        f"Using GPU {selected['index']} on device {device} "
        f"({selected['free']} MB free / {selected['total']} MB total, {selected['used']} MB used)"
    )
    return device


def adaptive_estimation(v_t, beta, square_term, i):
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    return v_t, (v_t / (1 - beta ** i))


def MFMC_t_trace(x, y, track_cov, i, cov_beta=0.95):
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    eps = 1e-6
    Rx = Rx + torch.eye(Rx.shape[0], device=Rx.device) * eps
    Ry = Ry + torch.eye(Ry.shape[0], device=Ry.device) * eps

    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, i)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, i)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, i)

    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    cost = -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T            + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T            - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T            + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T
    loss = -torch.trace(cost)
    return track_cov, loss


def tri_modal_projection_loss(fe1, fe2, fe3, proj12, proj23, proj13, trackers, step, cov_beta=0.5):
    concat_12 = torch.cat([fe1, fe2], dim=1)
    concat_23 = torch.cat([fe2, fe3], dim=1)
    concat_13 = torch.cat([fe1, fe3], dim=1)

    proj_12 = proj12(concat_12)
    proj_23 = proj23(concat_23)
    proj_13 = proj13(concat_13)

    trackers['track_1_23'], loss1 = MFMC_t_trace(fe1, proj_23, trackers['track_1_23'], step, cov_beta)
    trackers['track_2_13'], loss2 = MFMC_t_trace(fe2, proj_13, trackers['track_2_13'], step, cov_beta)
    trackers['track_3_12'], loss3 = MFMC_t_trace(fe3, proj_12, trackers['track_3_12'], step, cov_beta)
    return trackers, (loss1 + loss2 + loss3)


class ProjectionHead(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x


class NETWORK_F_MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.fc_list = nn.ModuleList()
        self.bn_list = nn.ModuleList()

        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)
        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x


class Advanced1DCNN_channel(nn.Module):
    def __init__(self, input_channels=1, num_classes=128, input_size=1280):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )

        feat_size = max(1, input_size // (4 * 4 * 4 * 4))
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(512, num_classes)
        self.MLP = NETWORK_F_MLP(
            input_dim=128 * input_channels,
            hidden_dim=4000,
            out_dim=num_classes,
            num_layers=1,
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]
        x = x.unsqueeze(2)
        x = x.flatten(0, 1)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.MLP(out)
        return out


class ComplexClassifier(nn.Module):
    def __init__(self, dim_features=128, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x


def get_factor_order(factor_name: str):
    return {
        'baseline': ['baseline'],
        'window_length': ['wl_2', 'wl_5', 'wl_10', 'wl_15'],
        'stride': ['stride_0p4', 'stride_1', 'stride_2', 'stride_10'],
        'normalization': ['norm_legacy', 'norm_firstsample', 'norm_zscore'],
        'outlier': ['thr_none', 'thr_3', 'thr_5', 'thr_7'],
    }[factor_name]


def sort_factor_frame(df: pd.DataFrame, factor_name: str) -> pd.DataFrame:
    order = get_factor_order(factor_name)
    working = df.copy()
    working['setting_name'] = pd.Categorical(working['setting_name'], categories=order, ordered=True)
    working = working.sort_values('setting_name').reset_index(drop=True)
    working['setting_name'] = working['setting_name'].astype(str)
    return working


def filter_factor_records(manifest_df: pd.DataFrame, factor_name: str) -> pd.DataFrame:
    factor_records = manifest_df.loc[manifest_df['factor'] == factor_name].copy()
    factor_records = sort_factor_frame(factor_records, factor_name)
    if SETTING_NAME_FILTER is not None:
        factor_records = factor_records.loc[factor_records['setting_name'].isin(SETTING_NAME_FILTER)].copy()
    if factor_records.empty:
        raise ValueError(f"No manifest records found for factor={factor_name} with the current filter.")
    return factor_records.reset_index(drop=True)


def load_setting_arrays(output_dir: Path):
    arrays = {
        'eeg': np.load(output_dir / 'eeg_data.npy').astype(np.float32),
        'eog': np.load(output_dir / 'eog_data.npy').astype(np.float32),
        'temp': np.load(output_dir / 'temp_data.npy').astype(np.float32),
        'labels': np.load(output_dir / 'emotion_labels.npy').astype(np.int64),
        'subject': np.load(output_dir / 'subject.npy').astype(np.int64),
    }
    return arrays


def create_models(eeg_shape, eog_shape, temp_shape, num_classes, device):
    net_eeg = Advanced1DCNN_channel(
        input_channels=eeg_shape[1],
        num_classes=128,
        input_size=eeg_shape[2],
    ).to(device)
    net_eog = Advanced1DCNN_channel(
        input_channels=eog_shape[1],
        num_classes=128,
        input_size=eog_shape[2],
    ).to(device)
    net_temp = Advanced1DCNN_channel(
        input_channels=temp_shape[1],
        num_classes=128,
        input_size=temp_shape[2],
    ).to(device)
    proj_12 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
    proj_23 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
    proj_13 = ProjectionHead(input_dim=256, hidden_dim=512, output_dim=128).to(device)
    classifier = ComplexClassifier(dim_features=128, num_classes=num_classes).to(device)
    return net_eeg, net_eog, net_temp, proj_12, proj_23, proj_13, classifier


def evaluate_model(net_eeg, classifier, test_eeg, test_labels, device, batch_size=100):
    net_eeg.eval()
    classifier.eval()
    preds = []
    refs = []
    with torch.no_grad():
        for start_idx in range(0, len(test_eeg), batch_size):
            end_idx = min(start_idx + batch_size, len(test_eeg))
            batch_eeg = test_eeg[start_idx:end_idx].to(device)
            batch_labels = test_labels[start_idx:end_idx].cpu().numpy()
            logits = classifier(net_eeg(batch_eeg))
            batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
            preds.append(batch_preds)
            refs.append(batch_labels)
    preds = np.concatenate(preds)
    refs = np.concatenate(refs)
    acc = accuracy_score(refs, preds)
    macro_f1 = f1_score(refs, preds, average='macro')
    return acc, macro_f1


def choose_best_result(best_acc, best_f1, acc, macro_f1):
    return (acc > best_acc) or (np.isclose(acc, best_acc) and macro_f1 > best_f1)


def plot_factor_results(df: pd.DataFrame, factor_name: str, figures_dir: Path, baseline_df: pd.DataFrame):
    ordered = sort_factor_frame(df, factor_name)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='white')
    x = np.arange(len(ordered))
    labels = ordered['setting_name'].tolist()
    metrics = [
        ('best_test_acc', 'Best Test Accuracy'),
        ('best_test_macro_f1', 'Best Test Macro-F1'),
    ]
    baseline_acc = float(baseline_df.iloc[0]['best_test_acc'])
    baseline_f1 = float(baseline_df.iloc[0]['best_test_macro_f1'])
    baseline_lookup = {
        'best_test_acc': baseline_acc,
        'best_test_macro_f1': baseline_f1,
    }

    for ax, (metric, ylabel) in zip(axes, metrics):
        ax.plot(x, ordered[metric].to_numpy(), marker='o', linewidth=1.8, color='#1f77b4')
        ax.scatter(x, ordered[metric].to_numpy(), color='#1f77b4', s=35)
        ax.axhline(baseline_lookup[metric], linestyle='--', linewidth=1.2, color='#7f7f7f', label='baseline')
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=25, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_xlabel('Setting')
        ax.set_title(f"{factor_name.replace('_', ' ').title()} sensitivity")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False)

    fig.tight_layout()
    png_path = figures_dir / f"{factor_name}_sensitivity.png"
    pdf_path = figures_dir / f"{factor_name}_sensitivity.pdf"
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    plt.show()
    plt.close(fig)
    return {'png': str(png_path), 'pdf': str(pdf_path)}


def plot_overview(results_by_factor, figures_dir: Path, mode_title: str):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor='white')
    factors = ['window_length', 'stride', 'normalization', 'outlier']
    for ax, factor_name in zip(axes.flatten(), factors):
        df = sort_factor_frame(results_by_factor[factor_name], factor_name)
        x = np.arange(len(df))
        ax.plot(x, df['best_test_acc'].to_numpy(), marker='o', linewidth=1.7, color='#1f77b4', label='Accuracy')
        ax.plot(x, df['best_test_macro_f1'].to_numpy(), marker='s', linewidth=1.4, color='#d62728', label='Macro-F1')
        ax.set_xticks(x)
        ax.set_xticklabels(df['setting_name'].tolist(), rotation=25, ha='right')
        ax.set_title(factor_name.replace('_', ' ').title())
        ax.set_ylim(0.0, 1.0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False)
    fig.suptitle(mode_title)
    fig.tight_layout()
    png_path = figures_dir / 'overall_sensitivity_overview.png'
    pdf_path = figures_dir / 'overall_sensitivity_overview.pdf'
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    plt.show()
    plt.close(fig)
    return {'png': str(png_path), 'pdf': str(pdf_path)}


def train_single_setting(setting_row: pd.Series, split_builder, device: torch.device) -> dict:
    output_dir = Path(setting_row['output_dir'])
    arrays = load_setting_arrays(output_dir)
    train_indices, test_indices, split_meta = split_builder(arrays)

    set_random_seed(RANDOM_SEED)

    train_eeg = torch.from_numpy(arrays['eeg'][train_indices]).float()
    test_eeg = torch.from_numpy(arrays['eeg'][test_indices]).float()
    train_eog = torch.from_numpy(arrays['eog'][train_indices]).float()
    train_temp = torch.from_numpy(arrays['temp'][train_indices]).float()
    train_labels = torch.from_numpy(arrays['labels'][train_indices]).long()
    test_labels = torch.from_numpy(arrays['labels'][test_indices]).long()

    num_classes = int(np.unique(arrays['labels']).shape[0])
    models = create_models(arrays['eeg'].shape, arrays['eog'].shape, arrays['temp'].shape, num_classes, device)
    net_eeg, net_eog, net_temp, proj_12, proj_23, proj_13, classifier = models

    feature_params = (
        list(net_eeg.parameters())
        + list(net_eog.parameters())
        + list(net_temp.parameters())
        + list(proj_12.parameters())
        + list(proj_23.parameters())
        + list(proj_13.parameters())
    )
    optimizer_features = optim.Adam(feature_params, lr=LEARNING_RATE_ENCODER, betas=(BETA1, BETA2), amsgrad=True)
    optimizer_classifier = optim.Adam(classifier.parameters(), lr=LEARNING_RATE_CLASSIFIER, betas=(BETA1, BETA2), amsgrad=True)

    if USE_CLASS_BALANCING:
        class_counts = torch.bincount(train_labels, minlength=num_classes).float()
        class_weights = 1.0 / torch.clamp(class_counts, min=1.0)
        class_weights = class_weights / class_weights.sum() * len(class_weights)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()

    feature_dim = 128
    trackers = {
        'track_1_23': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
        'track_2_13': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
        'track_3_12': {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        },
    }

    batch_size = min(BATCH_SIZE, len(train_eeg))
    best_acc = -1.0
    best_f1 = -1.0
    best_epoch = -1

    for iteration in range(1, TOTAL_ITERATIONS):
        optimizer_features.zero_grad()
        feature_batch_indices = torch.randint(0, len(train_eeg), (batch_size,))
        input_eeg = train_eeg[feature_batch_indices].to(device)
        input_eog = train_eog[feature_batch_indices].to(device)
        input_temp = train_temp[feature_batch_indices].to(device)

        feature_eeg = net_eeg(input_eeg)
        feature_eog = net_eog(input_eog)
        feature_temp = net_temp(input_temp)
        trackers, mfmc_loss = tri_modal_projection_loss(
            feature_eeg,
            feature_eog,
            feature_temp,
            proj_12,
            proj_23,
            proj_13,
            trackers,
            iteration,
            COV_BETA,
        )
        mfmc_loss.backward()
        optimizer_features.step()

        optimizer_classifier.zero_grad()
        classifier_batch_indices = torch.randint(0, len(train_eeg), (batch_size,))
        classifier_input = train_eeg[classifier_batch_indices].to(device)
        with torch.no_grad():
            classifier_feature = net_eeg(classifier_input)
        labels_batch = train_labels[classifier_batch_indices].to(device)
        logits = classifier(classifier_feature.detach())
        classifier_loss = criterion(logits, labels_batch)
        classifier_loss.backward()
        optimizer_classifier.step()

        if iteration % EVAL_INTERVAL == 0:
            acc, macro_f1 = evaluate_model(net_eeg, classifier, test_eeg, test_labels, device)
            if choose_best_result(best_acc, best_f1, acc, macro_f1):
                best_acc = float(acc)
                best_f1 = float(macro_f1)
                best_epoch = int(iteration)
            print(
                f"{setting_row['setting_name']} | iter={iteration:5d} | "
                f"best_acc={best_acc:.4f} | best_macro_f1={best_f1:.4f} | best_epoch={best_epoch}"
            )
            net_eeg.train()
            net_eog.train()
            net_temp.train()
            classifier.train()

    result = {
        'factor': setting_row['factor'],
        'setting_name': setting_row['setting_name'],
        'output_dir': setting_row['output_dir'],
        'window_length_sec': float(setting_row['window_length_sec']),
        'offset_sec': float(setting_row['offset_sec']),
        'stride_sec': float(setting_row['stride_sec']),
        'normalization': setting_row['normalization'],
        'outlier_threshold': setting_row['outlier_threshold'],
        'kept_windows': int(setting_row['kept_windows']),
        'removed_windows': int(setting_row['removed_windows']),
        'best_test_acc': best_acc,
        'best_test_macro_f1': best_f1,
        'best_epoch': best_epoch,
    }
    result.update(split_meta)
    return result


def run_experiments_for_records(records: pd.DataFrame, split_builder, device: torch.device) -> pd.DataFrame:
    rows = []
    for _, setting_row in records.iterrows():
        rows.append(train_single_setting(setting_row, split_builder, device))
    results_df = pd.DataFrame(rows)
    return sort_factor_frame(results_df, str(results_df.iloc[0]['factor']))


def export_summary_csvs(all_results: pd.DataFrame, summaries: dict, results_dir: Path, all_filename: str):
    all_results.to_csv(results_dir / all_filename, index=False)
    for name, df in summaries.items():
        df.to_csv(results_dir / name, index=False)


SUBJECT_INDEP_SPLIT = {
    "train_subjects": [17, 15, 4, 11, 5, 18, 20, 6, 13, 2, 22, 7, 9, 0, 3],
    "test_subjects": [14, 19, 12, 1],
}


def build_split(arrays):
    subject_ids = arrays['subject']
    train_mask = np.isin(subject_ids, SUBJECT_INDEP_SPLIT['train_subjects'])
    test_mask = np.isin(subject_ids, SUBJECT_INDEP_SPLIT['test_subjects'])
    train_indices = np.where(train_mask)[0]
    test_indices = np.where(test_mask)[0]
    if len(train_indices) == 0 or len(test_indices) == 0:
        raise ValueError('Subject-independent split produced an empty train or test partition.')
    split_meta = {
        'split_type': 'subject_independent',
        'train_subjects': json.dumps(SUBJECT_INDEP_SPLIT['train_subjects']),
        'test_subjects': json.dumps(SUBJECT_INDEP_SPLIT['test_subjects']),
    }
    return train_indices, test_indices, split_meta



In [ ]:
# Load manifest.csv

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing manifest: {MANIFEST_PATH}. Run DEAP_preprocess_sensitivity.py first inside the MFMC environment."
    )

manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df = manifest_df.sort_values(['factor', 'setting_name']).reset_index(drop=True)
DEVICE = select_device()

print(f"Loaded {len(manifest_df)} preprocessing settings from manifest.csv")
display(manifest_df)


In [ ]:
# Window-length sensitivity

baseline_records = filter_factor_records(manifest_df, 'baseline')
window_length_records = filter_factor_records(manifest_df, 'window_length')

baseline_results = run_experiments_for_records(baseline_records, build_split, DEVICE)
window_length_results = run_experiments_for_records(window_length_records, build_split, DEVICE)

display(baseline_results)
display(window_length_results)


In [ ]:
# Stride sensitivity

stride_records = filter_factor_records(manifest_df, 'stride')
stride_results = run_experiments_for_records(stride_records, build_split, DEVICE)
display(stride_results)


In [ ]:
# Normalization sensitivity

normalization_records = filter_factor_records(manifest_df, 'normalization')
normalization_results = run_experiments_for_records(normalization_records, build_split, DEVICE)
display(normalization_results)


In [ ]:
# Outlier-threshold sensitivity

outlier_records = filter_factor_records(manifest_df, 'outlier')
outlier_results = run_experiments_for_records(outlier_records, build_split, DEVICE)
display(outlier_results)


In [ ]:
# Export CSV summaries

results_by_factor = {
    'baseline': baseline_results,
    'window_length': window_length_results,
    'stride': stride_results,
    'normalization': normalization_results,
    'outlier': outlier_results,
}

all_results = pd.concat([
    baseline_results,
    window_length_results,
    stride_results,
    normalization_results,
    outlier_results,
], ignore_index=True)

summary_window_length = window_length_results.copy()
summary_stride = stride_results.copy()
summary_normalization = normalization_results.copy()
summary_outlier = outlier_results.copy()

export_summary_csvs(
    all_results=all_results,
    summaries={
        'summary_window_length.csv': summary_window_length,
        'summary_stride.csv': summary_stride,
        'summary_normalization.csv': summary_normalization,
        'summary_outlier.csv': summary_outlier,
    },
    results_dir=RESULTS_DIR,
    all_filename='all_indep_results.csv',
)

print(f"Saved CSV summaries to {RESULTS_DIR}")
display(all_results)


In [ ]:
# Export figures

figure_exports = {
    'window_length': plot_factor_results(window_length_results, 'window_length', FIGURES_DIR, baseline_results),
    'stride': plot_factor_results(stride_results, 'stride', FIGURES_DIR, baseline_results),
    'normalization': plot_factor_results(normalization_results, 'normalization', FIGURES_DIR, baseline_results),
    'outlier': plot_factor_results(outlier_results, 'outlier', FIGURES_DIR, baseline_results),
    'overview': plot_overview({
        'window_length': window_length_results,
        'stride': stride_results,
        'normalization': normalization_results,
        'outlier': outlier_results,
    }, FIGURES_DIR, 'Subject-Independent preprocessing sensitivity'),
}

pd.DataFrame(figure_exports).T
